# 05 - Learned prior, physics kept: Plug-and-Play (PnP)

Lecture section: 4.8  |  Spine term this tutorial changes: the prior $R$ becomes a *pretrained denoiser*

$$\hat{x} = \arg\min_x\ \underbrace{D(Ax, y)}_{\text{data fidelity (UNCHANGED)}} + \underbrace{R(x)}_{\text{prior: TV} \,\rightarrow\, \text{learned denoiser}}$$

Same problem, same physics, same data-fidelity as notebooks 3 and 4 (40-view noisy CT).
The ONLY thing we change is the prior $R$: instead of a hand-crafted regularizer
(Tikhonov, TV) we plug in a **pretrained image denoiser** (DRUNet). This is the
**Plug-and-Play (PnP)** idea: a proximal algorithm calls $\mathrm{prox}_R$ at every step,
and a denoiser *is* an excellent learned proximal operator -- so we drop it in. We solve
with **PGD** and finish with **DPIR**, a ready-made PnP preset.

In [1]:
import math
import tutorial_common as tc
import deepinv as dinv
import torch

tc.set_seed()

deepinv 0.4.1 | torch 2.9.1 | device cpu


## The shared problem (identical to notebooks 3 & 4)

Data-fidelity $D$ and physics $A$ are fixed: 40-angle sparse-view CT with Gaussian
noise ($\sigma=0.02$). This is our PAT/CT stand-in: linear, ill-posed, limited-view.
Only $R$ changes below.

In [2]:
angles, sigma = 40, 0.02
phys = tc.ct_physics(angles=angles, sigma=sigma, size=128)  # the fixed physics A (+ noise)
x = tc.load_hero(128)                                       # ground-truth object x, (1,1,128,128)
y = phys(x)                                                 # noisy measurements y = A x + noise

data_fidelity = dinv.optim.L2()        # D(Ax, y) = 0.5 ||Ax - y||^2  -- SAME as notebook 4
stepsize = tc.stepsize_for(phys, x)    # 1 / ||A^T A|| ~ 1, exactly as in notebook 4

# No-prior baseline: filtered back-projection (the same FBP used in notebooks 0 and 3).
x_fbp = phys.fbp(y)
print(f"FBP (no prior)    PSNR {tc.psnr(x_fbp, x):.1f} dB")

# The learned solver warm-starts from the rescaled back-projection.
scaling = math.pi / (2 * angles)
init = lambda y, physics: {"est": (physics.A_adjoint(y) * scaling,
                                   physics.A_adjoint(y) * scaling)}

FBP (no prior)    PSNR 15.2 dB


## Baseline prior: TV (exactly the tuned reconstruction from notebook 4)

We re-run notebook 4's TV with the *same* hyper-parameters, so the comparison is honest:
same $D$, same $A$, same problem. Remember the `prior=...` line below -- in a moment we
change *only* it.

In [3]:
tv_model = dinv.optim.optim_builder(
    iteration="PGD",
    prior=dinv.optim.TVPrior(n_it_max=20),     # <-- hand-crafted prior R = total variation
    data_fidelity=data_fidelity,               # SAME D
    params_algo={"stepsize": stepsize, "lambda": 0.001},   # identical to notebook 4
    max_iter=300, verbose=False,
)
x_tv = tv_model(y, phys)
print(f"TV                PSNR {tc.psnr(x_tv, x):.1f} dB")

TV                PSNR 21.4 dB


## Learned prior: Plug-and-Play with a pretrained DRUNet

A denoiser $\mathrm{D}_\sigma$ maps a noisy image to a clean one -- exactly what
$\mathrm{prox}_R$ does in a proximal step. **PnP** replaces the prox of the hand-crafted
$R$ by a call to a **pretrained denoiser**. We load DRUNet (grayscale, 1-channel) with
weights from the deepinv model zoo -- **no training on our CT problem**. Its strength is
the `g_param` (the denoiser's internal noise level).

In [4]:
# Grayscale DRUNet: the 1-channel pretrained weights download and load cleanly.
denoiser = dinv.models.DRUNet(in_channels=1, out_channels=1,
                              pretrained="download", device=tc.DEVICE)

pnp_model = dinv.optim.optim_builder(
    iteration="PGD",
    prior=dinv.optim.PnP(denoiser=denoiser),   # <-- learned prior R = pretrained denoiser
    data_fidelity=data_fidelity,               # SAME D as TV above
    params_algo={"stepsize": 2.0, "g_param": 0.01},  # g_param = denoiser noise level
    max_iter=80, early_stop=False, verbose=False, custom_init=init,
)
pnp_model.eval()
with torch.no_grad():                          # inference only: no gradients needed
    x_pnp = pnp_model(y, phys)
print(f"PnP (PGD+DRUNet)  PSNR {tc.psnr(x_pnp, x):.1f} dB")

PnP (PGD+DRUNet)  PSNR 24.2 dB


### A note on RED

**RED** (Regularization by Denoising, `deepinv.optim.RED`) uses the *same* denoiser, but
inside the **gradient** rather than the prox: it adds a term whose gradient is
$\lambda\,(x - \mathrm{D}_\sigma(x))$, pulling each iterate toward its own denoised
version. It's the gradient-based sibling of PnP; on this CT problem PnP (and the DPIR
preset below) gave the strongest results, so those are what we show. Swapping
`PnP(denoiser=...)` for `RED(denoiser=...)` is, again, a one-line change of $R$.

## DPIR: the same idea as a one-liner preset

**DPIR** is Plug-and-Play wrapped up: a half-quadratic-splitting schedule with DRUNet and
a decreasing denoiser strength, all preset for you. We only must pass the **grayscale**
denoiser (the default 3-channel DRUNet crashes on our 1-channel CT image). It needs `sigma`
and the measurements -- no ground truth, no tuning.

In [5]:
dpir = dinv.optim.DPIR(
    sigma=sigma,
    denoiser=dinv.models.DRUNet(in_channels=1, out_channels=1,
                                pretrained="download", device=tc.DEVICE),
    device=tc.DEVICE,
)
with torch.no_grad():
    x_dpir = dpir(y, phys)
print(f"DPIR (preset)     PSNR {tc.psnr(x_dpir, x):.1f} dB")

DPIR (preset)     PSNR 25.1 dB


## Comparison: same physics $D$, smarter prior $R$

Left to right the prior gets smarter while $D$ and $A$ stay fixed. The learned-prior
reconstructions (PnP, DPIR) are visibly cleaner and jump well above the tuned TV baseline.

In [6]:
tc.save_images(
    [x, x_fbp, x_tv, x_pnp, x_dpir],
    titles=[
        "x (ground truth)",
        tc.title_psnr("FBP (no prior)", x_fbp, x),
        tc.title_psnr("TV prior", x_tv, x),
        tc.title_psnr("PnP (DRUNet)", x_pnp, x),
        tc.title_psnr("DPIR (preset)", x_dpir, x),
    ],
    fname="05_comparison.png",
    suptitle="Same data-fidelity D and physics A — only the prior R changes",
)

saved /Users/jonathan/Code/deepinv/lecture-tutorials/figures/05_comparison.png


## The punchline: it's a one-line swap

Going from notebook 4 (TV) to here (PnP) is **literally** replacing the prior argument
inside the *same* `optim_builder`, with the *same* `data_fidelity=L2()` and the *same*
physics `phys`:

$$\hat{x} = \arg\min_x\ \underbrace{\tfrac12\|Ax - y\|^2}_{D:\ \text{unchanged}} + \underbrace{R(x)}_{R = \text{learned denoiser}}$$

DPIR is just this idea shipped as a ready-made preset.

In [7]:
# notebook 4:  prior=dinv.optim.TVPrior(n_it_max=20)          # hand-crafted prior
# notebook 5:  prior=dinv.optim.PnP(denoiser=DRUNet(...))     # learned prior
# ...everything else (data_fidelity=L2(), physics phys, the solver) is identical.
print("TV -> PnP is a one-line change of R; D and A are untouched.")
print(f"  FBP {tc.psnr(x_fbp,x):5.1f} | TV {tc.psnr(x_tv,x):5.1f} | "
      f"PnP {tc.psnr(x_pnp,x):5.1f} | DPIR {tc.psnr(x_dpir,x):5.1f}  dB")

TV -> PnP is a one-line change of R; D and A are untouched.
  FBP  15.2 | TV  21.4 | PnP  24.2 | DPIR  25.1  dB


## Takeaway

Keep the physics-driven data-fidelity $D$ exactly as is, and swap the hand-crafted prior
$R$ for a **pretrained denoiser** -- and you get a large PSNR jump and visibly cleaner
images, **with no training on this specific inverse problem**.